In [1]:
# Install packages
!pip install -q transformers sentencepiece

import warnings
warnings.filterwarnings('ignore')

from transformers import MarianMTModel, MarianTokenizer
import pandas as pd
import torch
from tqdm import tqdm

print("="*60)
print("AKKADIAN TRANSLATION - INFERENCE ONLY")
print("="*60)

# Configuration
MAX_LENGTH = 200

# Load YOUR pre-trained model
print("\n[1/3] Loading fine-tuned model...")
# IMPORTANT: Update this path after you add your dataset
MODEL_PATH = "/kaggle/input/akkadian-translation-project/model/checkpoint-528"

tokenizer = MarianTokenizer.from_pretrained(MODEL_PATH)
model = MarianMTModel.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print(f"Device: {device}")
print("✅ Model loaded successfully!")

# Load test data
print("\n[2/3] Loading test data...")
test_df = pd.read_csv('/kaggle/input/deep-past-initiative-machine-translation/test.csv')
print(f"Test samples: {len(test_df)}")

# Translate function
def translate_batch(texts, batch_size=16):
    translations = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", max_length=MAX_LENGTH, 
                          truncation=True, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=MAX_LENGTH)
        
        translations.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    
    return translations

# Generate predictions
print("\n[3/3] Generating predictions...")
predictions = translate_batch(test_df['transliteration'].tolist())

# Create submission
submission_df = pd.DataFrame({
    'id': test_df['id'], 
    'translation': predictions
})

submission_df.to_csv('submission.csv', index=False)

print("\n" + "="*60)
print("✅ DONE! submission.csv created")
print("="*60)
print(f"Rows: {len(submission_df)}")
print("\nFirst 3 predictions:")
print(submission_df.head(3))

2026-02-03 08:18:12.468265: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770106692.672202      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770106692.725397      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770106693.171980      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770106693.172020      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770106693.172023      55 computation_placer.cc:177] computation placer alr

AKKADIAN TRANSLATION - INFERENCE ONLY

[1/3] Loading fine-tuned model...
Device: cuda
✅ Model loaded successfully!

[2/3] Loading test data...
Test samples: 4

[3/3] Generating predictions...


Translating: 100%|██████████| 1/1 [00:02<00:00,  2.52s/it]


✅ DONE! submission.csv created
Rows: 4

First 3 predictions:
   id                                        translation
0   0  From the Kanesh colony to Aqil ... the dagger ...
1   1  From the dagger to the City, the man from Man-...
2   2  Say to the representatives of Ta'am-Anni, eith...


In [4]:
!ls -la /kaggle/input/akkadian-translation-project/model/checkpoint-528/

total 910928
drwxr-xr-x 2 nobody nogroup         0 Feb  3 08:13 .
drwxr-xr-x 3 nobody nogroup         0 Feb  3 08:13 ..
-rw-r--r-- 1 nobody nogroup      1257 Feb  3 08:13 config.json
-rw-r--r-- 1 nobody nogroup       277 Feb  3 08:13 generation_config.json
-rw-r--r-- 1 nobody nogroup 309965092 Feb  3 08:13 model.safetensors
-rw-r--r-- 1 nobody nogroup 623763595 Feb  3 08:13 optimizer.pt
-rw-r--r-- 1 nobody nogroup     14645 Feb  3 08:13 rng_state.pth
-rw-r--r-- 1 nobody nogroup      1383 Feb  3 08:13 scaler.pt
-rw-r--r-- 1 nobody nogroup      1465 Feb  3 08:13 scheduler.pt
-rw-r--r-- 1 nobody nogroup    800087 Feb  3 08:13 source.spm
-rw-r--r-- 1 nobody nogroup        74 Feb  3 08:13 special_tokens_map.json
-rw-r--r-- 1 nobody nogroup    779494 Feb  3 08:13 target.spm
-rw-r--r-- 1 nobody nogroup      1070 Feb  3 08:13 tokenizer_config.json
-rw-r--r-- 1 nobody nogroup      3686 Feb  3 08:13 trainer_state.json
-rw-r--r-- 1 nobody nogroup      5969 Feb  3 08:13 training_args.bin
-rw-r--r-